# Phase 4 Evaluation, Explainability, and Safety Checks

Phase 4 validates the saved Phase 3 models for real-world hospital use. It evaluates training and test performance, highlights business-critical recall, reviews feature importance, checks performance across demographic and regional segments, and creates governance-ready model-card outputs.

## Business Goal

Ensure that predictions are reliable, interpretable, and safe for hospital operations and finance teams.

Critical business metrics:

- Visit risk model: recall for the **High** risk class.
- Claim outcome model: recall for the **Rejected** claim class.

These metrics matter because missing high-risk visits can harm operational triage, while missing rejected claims can reduce proactive revenue-cycle control.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PHASE4_DIR = ROOT / "docs" / "phase4"
MODEL_TABLE_PATH = ROOT / "data_outputs" / "model_table.csv"

df = pd.read_csv(MODEL_TABLE_PATH, parse_dates=["visit_date", "billing_date", "registration_date"])
df.shape

(25000, 44)

## Generate Evaluation Reports

The evaluation script loads the saved best models, applies the same time-based split as Phase 3, and writes model evaluation reports, segment metrics, feature importance files, and a consolidated model card.

In [2]:
from scripts.evaluate_phase4_models import main as evaluate_phase4_models

evaluate_phase4_models()

      model  test_accuracy  test_macro_f1 critical_class  critical_class_recall
 risk_model         0.4092         0.3485           High                 0.1261
claim_model         0.4444         0.3743       Rejected                 0.6044
wrote reports to: /home/suuri/Downloads/capstone-iitm/docs/phase4


In [3]:
summary = pd.read_csv(PHASE4_DIR / "phase4_evaluation_summary.csv")
summary

,model,test_accuracy,test_macro_f1,critical_class,critical_class_recall
0,risk_model,0.4092,0.3485,High,0.1261
1,claim_model,0.4444,0.3743,Rejected,0.6044


## Evaluation Summary

| model       |   test_accuracy |   test_macro_f1 | critical_class   |   critical_class_recall |
|:------------|----------------:|----------------:|:-----------------|------------------------:|
| risk_model  |          0.4092 |          0.3485 | High             |                  0.1261 |
| claim_model |          0.4444 |          0.3743 | Rejected         |                  0.6044 |

Business interpretation:

- The risk model's High-class recall is low, so it should not be used as an autonomous safety screen.
- The claim model captures a stronger share of Rejected claims and is more immediately useful for billing review queues.
- Macro F1 remains modest for both models, which supports using the models as prioritization tools rather than final decision engines.

![Phase 4 test performance](../reports/figures/phase4_test_performance.png)

## Train vs Test Governance Metrics

The JSON metrics file stores train and test accuracy, macro F1, weighted F1, and critical-class recall. Reviewing train and test together helps identify overfitting and production-readiness risk.

In [4]:
metrics = json.loads((PHASE4_DIR / "phase4_metrics.json").read_text())
metrics

{'risk_model': {'critical_class': 'High',
  'train': {'accuracy': 0.8023,
   'macro_f1': 0.8037,
   'weighted_f1': 0.8019,
   'critical_class_recall': 0.8295},
  'test': {'accuracy': 0.4092,
   'macro_f1': 0.3485,
   'weighted_f1': 0.4013,
   'critical_class_recall': 0.1261},
  'fairness_gaps': [{'segment': 'city',
    'min_critical_recall': 0.0599,
    'max_critical_recall': 0.2364,
    'critical_recall_gap': 0.1765,
    'lowest_recall_group': 'Mumbai'},
   {'segment': 'gender',
    'min_critical_recall': 0.1136,
    'max_critical_recall': 0.1394,
    'critical_recall_gap': 0.0258,
    'lowest_recall_group': 'F'},
   {'segment': 'insurance_provider',
    'min_critical_recall': 0.0887,
    'max_critical_recall': 0.1606,
    'critical_recall_gap': 0.0719,
    'lowest_recall_group': 'CareOne'}]},
 'claim_model': {'critical_class': 'Rejected',
  'train': {'accuracy': 0.6669,
   'macro_f1': 0.6545,
   'weighted_f1': 0.6956,
   'critical_class_recall': 0.8954},
  'test': {'accuracy': 0.4444

## Fairness and Segment Review

Segment checks are computed across `gender`, `city`, and `insurance_provider`. The key safety signal is the gap between the lowest and highest critical-class recall within each segment type.

In [5]:
risk_segments = pd.read_csv(PHASE4_DIR / "risk_model_segment_metrics.csv")
claim_segments = pd.read_csv(PHASE4_DIR / "claim_model_segment_metrics.csv")

risk_segments.head(10), claim_segments.head(10)

(              segment       value  rows  accuracy  macro_f1 critical_class  \
 0                city      Mumbai   816    0.4044    0.3229           High   
 1                city     Chennai   824    0.4260    0.3418           High   
 2                city   Bangalore   829    0.4499    0.3445           High   
 3                city   Hyderabad   893    0.3785    0.3315           High   
 4                city       Delhi   849    0.4040    0.3620           High   
 5                city        Pune   789    0.3942    0.3596           High   
 6              gender           F  2530    0.4099    0.3456           High   
 7              gender           M  2470    0.4085    0.3513           High   
 8  insurance_provider     CareOne  1235    0.4405    0.3536           High   
 9  insurance_provider  SecureLife  1200    0.4033    0.3395           High   
 
    critical_class_support  critical_class_recall  
 0                     167                 0.0599  
 1                     16

Risk model lowest recall segments:

| segment   | value     |   rows |   accuracy |   macro_f1 | critical_class   |   critical_class_support |   critical_class_recall |
|:----------|:----------|-------:|-----------:|-----------:|:-----------------|-------------------------:|------------------------:|
| city      | Mumbai    |    816 |     0.4044 |     0.3229 | High             |                      167 |                  0.0599 |
| city      | Chennai   |    824 |     0.426  |     0.3418 | High             |                      167 |                  0.0599 |
| city      | Bangalore |    829 |     0.4499 |     0.3445 | High             |                      151 |                  0.0728 |
| city      | Hyderabad |    893 |     0.3785 |     0.3315 | High             |                      189 |                  0.127  |
| city      | Delhi     |    849 |     0.404  |     0.362  | High             |                      184 |                  0.1902 |
| city      | Pune      |    789 |     0.3942 |     0.3596 | High             |                      165 |                  0.2364 |
| gender    | F         |   2530 |     0.4099 |     0.3456 | High             |                      528 |                  0.1136 |
| gender    | M         |   2470 |     0.4085 |     0.3513 | High             |                      495 |                  0.1394 |

Claim model lowest recall segments:

| segment   | value     |   rows |   accuracy |   macro_f1 | critical_class   |   critical_class_support |   critical_class_recall |
|:----------|:----------|-------:|-----------:|-----------:|:-----------------|-------------------------:|------------------------:|
| city      | Chennai   |    824 |     0.443  |     0.3851 | Rejected         |                      134 |                  0.5448 |
| city      | Mumbai    |    816 |     0.4461 |     0.3713 | Rejected         |                      127 |                  0.5591 |
| city      | Hyderabad |    893 |     0.4726 |     0.3916 | Rejected         |                      142 |                  0.6056 |
| city      | Bangalore |    829 |     0.4427 |     0.3813 | Rejected         |                      126 |                  0.627  |
| city      | Pune      |    789 |     0.4094 |     0.3396 | Rejected         |                       94 |                  0.6489 |
| city      | Delhi     |    849 |     0.4488 |     0.3737 | Rejected         |                      105 |                  0.6667 |
| gender    | M         |   2470 |     0.4348 |     0.3677 | Rejected         |                      335 |                  0.603  |
| gender    | F         |   2530 |     0.4538 |     0.3805 | Rejected         |                      393 |                  0.6056 |

![Phase 4 fairness segments](../reports/figures/phase4_fairness_segments.png)

## Explainability Summary

Random Forest feature importance is used as the Phase 4 explainability baseline. These values describe predictive contribution in the fitted model; they do not prove causality.

In [ ]:
risk_importance = pd.read_csv(PHASE4_DIR / "risk_model_feature_importance.csv")
claim_importance = pd.read_csv(PHASE4_DIR / "claim_model_feature_importance.csv")

risk_importance.head(10), claim_importance.head(10)

Risk model top drivers:

| feature                              |   importance | source_feature              |
|:-------------------------------------|-------------:|:----------------------------|
| numeric__length_of_stay_hours_filled |     0.165998 | length_of_stay_hours_filled |
| numeric__days_since_registration     |     0.122919 | days_since_registration     |
| numeric__doctor_id                   |     0.115506 | doctor_id                   |
| numeric__age                         |     0.109824 | age                         |
| numeric__visit_frequency             |     0.064177 | visit_frequency             |
| numeric__visit_month                 |     0.060147 | visit_month                 |
| numeric__visit_day_of_week           |     0.056127 | visit_day_of_week           |
| numeric__visit_quarter               |     0.029522 | visit_quarter               |
| numeric__chronic_flag                |     0.021069 | chronic_flag                |
| categorical__gender_M                |     0.016373 | gender_M                    |

Claim model top drivers:

| feature                              |   importance | source_feature              |
|:-------------------------------------|-------------:|:----------------------------|
| numeric__billed_amount               |     0.279473 | billed_amount               |
| numeric__length_of_stay_hours_filled |     0.078779 | length_of_stay_hours_filled |
| numeric__billing_lag_days            |     0.077019 | billing_lag_days            |
| numeric__days_since_registration     |     0.076857 | days_since_registration     |
| numeric__doctor_id                   |     0.071197 | doctor_id                   |
| numeric__age                         |     0.070415 | age                         |
| numeric__billing_month               |     0.038511 | billing_month               |
| numeric__visit_frequency             |     0.037579 | visit_frequency             |
| numeric__visit_month                 |     0.035661 | visit_month                 |
| numeric__visit_day_of_week           |     0.03352  | visit_day_of_week           |

![Phase 4 feature importance](../reports/figures/phase4_feature_importance.png)

## Imbalance Handling Recommendations

The current models already use class weighting and balanced subsampling. The next iteration should test stronger imbalance treatments:

- Hyperparameter tuning: adjust class weights, tree depth, minimum leaf size, and number of estimators.
- Feature engineering: add interaction features such as billed amount relative to department averages and payer-specific historical rejection patterns.
- Resampling: evaluate SMOTE, undersampling, or hybrid resampling for minority classes, especially Rejected claims and High-risk visits.

These should be tested with the same time-based split to avoid overly optimistic results.

## Model Card and Report Outputs

Phase 4 writes the following governance artifacts:

- `docs/phase4/risk_model_evaluation_report.md`
- `docs/phase4/claim_model_evaluation_report.md`
- `docs/phase4/model_card.md`
- `docs/phase4/explainability_summary.md`
- `docs/phase4/phase4_metrics.json`
- `docs/phase4/phase4_evaluation_summary.csv`

The model card documents intended use, limitations, assumptions, fairness review, and monitoring recommendations.

In [ ]:
print((PHASE4_DIR / "model_card.md").read_text()[:1500])

## Business Conclusion

The claim outcome model is the stronger near-term candidate for workflow prioritization because Rejected-claim recall is materially higher than High-risk recall in the visit model. The visit-risk model should remain a monitoring and experimentation artifact until recall for the High class improves. Both models require human review, segment monitoring, and periodic retraining before production deployment.